In [ ]:
import functools

import matplotlib.pyplot as plt
import numpy as np
import torch
from network import Net

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

In [ ]:
def grad(outputs, inputs):
    """Computes the partial derivative of an output with respect to an input.
    Args:
        outputs: (N, 1) tensor
        inputs: (N, D) tensor
    """
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)

In [ ]:
def cooling_law(time, Tenv, T0, R):
    T = Tenv + (T0 - Tenv) * np.exp(-R * time)
    return T

In [ ]:
np.random.seed(10)

Tenv = 25
T0 = 100
R = 0.005
times = np.linspace(0, 1000, 1000)
eq = functools.partial(cooling_law, Tenv=Tenv, T0=T0, R=R)
temps = eq(times)

# Make training data
t = np.linspace(0, 300, 15)
T = eq(t) + 2 * np.random.randn(15)

plt.plot(times, temps)
plt.plot(t, T, "o")
plt.legend(["Equation", "Training data"])
plt.ylabel("Temperature (C)")
plt.xlabel("Time (s)")

Simple feedforward neural network

In [ ]:
def l2_reg(model: torch.nn.Module):
    return torch.sum(sum([p.pow(2.0) for p in model.parameters()]))

In [ ]:
net = Net(1, 1, loss2=l2_reg, epochs=20000, lr=1e-4, loss2_weight=1).to(DEVICE)

losses = net.fit(t, T)

plt.plot(losses)
plt.yscale("log")

In [ ]:
preds = net.predict(times)
plt.plot(times, temps, alpha=0.8)
plt.plot(t, T, "o")
plt.plot(times, preds, alpha=0.8)

plt.legend(labels=["Equation", "Training data", "Network"])
plt.ylabel("Temperature (C)")
plt.xlabel("Time (s)")

Physics informed neural networks

In [ ]:
def physics_loss_10(model: torch.nn.Module):
    ts = torch.linspace(0, 1000, steps=10).view(-1, 1).requires_grad_(True).to(DEVICE)
    temps = model(ts)
    dT = grad(temps, ts)[0]
    pde = R * (Tenv - temps) - dT

    return torch.mean(pde**2)

In [ ]:
net_1000 = Net(1, 1, loss2=physics_loss_10, epochs=30000, loss2_weight=1, lr=1e-5).to(DEVICE)

losses_1000 = net_1000.fit(t, T)
plt.plot(losses_1000)
plt.yscale("log")

In [ ]:
colloc_points = np.linspace(0, 1000, 10)
colloc_temps = eq(colloc_points)

In [ ]:
preds = net_1000.predict(times)

plt.plot(times, temps, alpha=0.8)
plt.plot(t, T, "o")
plt.plot(times, preds, alpha=0.8)
plt.plot(colloc_points, colloc_temps, "x")
plt.legend(labels=["Equation", "Training data", "PINN", "Collocation points"])
plt.ylabel("Temperature (C)")
plt.xlabel("Time (s)")